<a href="https://colab.research.google.com/github/namigabbasov/text-analysis-and-nlp/blob/main/curate_faculty_pubs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Install and Load Libraries

In [ ]:
!pip install -q pandas numpy Unidecode rapidfuzz langdetect sentence-transformers seaborn graphviz requests
!pip install -q python-dateutil openai tqdm transformers torch sentencepiece

from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

import pandas as pd
import numpy as np
import re

from tqdm import tqdm
import openai

from unidecode import unidecode
from dateutil.parser import parse

from rapidfuzz import fuzz, process

from langdetect import detect

from sentence_transformers import SentenceTransformer

import matplotlib.pyplot as plt
import seaborn as sns

import requests

from graphviz import Digraph

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 12.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 57.0 MB/s eta 0:00:00


## Load Data

In [ ]:
sustainability_model = {
    "Automated (60%)": [
        "Text normalization",
        "Date parsing & standardization",
        "Name parsing & normalization",
        "Deduplication",
        "Publication type mapping",
        "Full-text search indexing",
        "Discovery: LLMs + retrieval connected via MCP"

    ],
    "Human-in-the-Loop (30%)": [
        "Curator review of duplicate merge decisions",
        "Approval of Bluebook citations",
        "Handling of flagged/unusual records",


    ],
    "Manual (10%)": [
        "Resolve author disambiguation",
        "Verify API enrichment errors",
        "Non-standard publication formats",
        "Error correction & feedback",
    ],
}

In [ ]:
df = pd.read_csv("df.csv")

## Initial Data Check

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1927 entries, 0 to 1926
Data columns (total 31 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   ID                          1926 non-null   float64
 1   Faculty Contributor         1927 non-null   object 
 2   Faculty Co-Authors/Editors  96 non-null     object 
 3   Co-Authors                  560 non-null    object 
 4   Title                       1927 non-null   object 
 5   Publication Type            1927 non-null   object 
 6   Contribution                1699 non-null   object 
 7   Date                        1927 non-null   object 
 8   Year                        1926 non-null   float64
 9   Status                      1316 non-null   object 
 10  Serial Title                1440 non-null   object 
 11  Publisher                   1680 non-null   object 
 12  Publication Title           251 non-null    object 
 13  Editor(s)                   127 n

In [ ]:
df.columns

Index(['ID', 'Faculty Contributor', 'Faculty Co-Authors/Editors', 'Co-Authors',
       'Title', 'Publication Type', 'Contribution', 'Date', 'Year', 'Status',
       'Serial Title', 'Publisher', 'Publication Title', 'Editor(s)',
       'Citation', 'Italics', 'Volume', 'Issue', 'Pages', 'Edition', 'ISBN',
       'ISSN', 'Abstract', 'Source Link(s)', 'Stanford Link', 'DOI', 'PURL',
       'SSRN', 'In ORCID', 'Corrected', 'Designation'],
      dtype='object')

In [ ]:
df.isnull().sum()

,0
ID,1
Faculty Contributor,0
Faculty Co-Authors/Editors,1831
Co-Authors,1367
Title,0
Publication Type,0
Contribution,228
Date,0
Year,1
Status,611


## Cheching Potential Issues with Faculty Contributor Names

In [ ]:
### Display ALL faculty names and how many times they appear

freq = df["Faculty Contributor"].value_counts()
freq_table = freq.to_frame().reset_index()
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

display(freq_table)

,Faculty Contributor,count
0,"Mello, Michelle M.",116
1,"Ho, Daniel E.",114
2,"Lemley, Mark A.",112
3,"Ouellette, Lisa Larrimore",93
4,"Greely, Henry T.",70
5,"McConnell, Michael W.",67
6,"Studdert, David M.",65
7,"Karlan, Pamela S.",56
8,"Engstrom, Nora Freeman",56
9,"Douek, Evelyn",53


## Checking Titles

Below, I run several pieces of code to check the titles. In addition to some obvious issues (HTML pollution, publication type given as a title, etc.), others reveal hidden formatting errors—such as using “Crimmigration,” instead of “Crimmigration:” or inserting extra spaces. All of these checks could be incorporated into a single Python function that processes titles to flag problematic cases for human review. This could also support the development of a full, deterministic pipeline to automatically correct formatting issues.

In [ ]:
### How many titles are unique
##### This will need further review because some of titles are the same not because they are the same publicaiton but because there are issues in titles.
### For example, some titles are "Book Review" or journal names such "Nature". These are not the same publications
TITLE_COL = "Title"
titles = df[TITLE_COL].astype(str).fillna("").str.strip()
print("Total rows:", len(titles))
print("Unique titles:", titles.nunique())

Total rows: 1927
Unique titles: 1741


In [ ]:
def print_list(title, lst):
    print(f"\n===== {title} (count = {len(lst)}) =====")
    for item in lst:
        print(item)

In [ ]:
### Leading punctuation or quotes
### This helps to check if punctuation or quotes are misplaced.
### For example, "Crimmigration," should actually be "Crimmigration:"
leading = titles[titles.str.match(r'^[\"\'“”‘’\[\({]')].tolist()
print_list("Titles with LEADING punctuation/quotes", leading)


===== Titles with LEADING punctuation/quotes (count = 9) =====
"With the Indian Tribes": Race, Citizenship, and Original Constitutional Meanings
"Crimmigration," Race, and Critical Race Theory in the United States
"Falsely Shouting Fire"
"What Kind of Oversight Board Have You Given Us"
"Courthouse AI" and Access to Justice in the U.S.
"Staying in the Takings Lane: The Compensation Issue in Cedar Point Nursery," Cardozo Law de Novo 90 (2022)
"Stock Drop" Lawsuits
"Partly Federal, Partly National": The Founders' Middle Course
"Vaccine Passport" Certification — Policy and Ethical Considerations


In [ ]:
trailing = titles[titles.str.match(r'[\"\'”’\]\)}]$')].tolist()
print_list("Titles with TRAILING punctuation/quotes", trailing)


===== Titles with TRAILING punctuation/quotes (count = 0) =====


In [ ]:
### Titles containing bracketed material
brackets = titles[titles.str.contains(r'[\(\[\{].*[\)\]\}]')].tolist()
print_list("Titles containing BRACKETS", brackets)



===== Titles containing BRACKETS (count = 62) =====
We the (Native) People?: How Indigenous Peoples Debated the U.S. Constitution
Justices argue over text (and ceviche) in ruling that Alaska Native corporations are "Indian tribes"
Opinion Analysis (Herrera v. Wyoming): Court Rejects Issue Preclusion in Affirming Crow Tribe's Treaty Hunting Right
Transnational Racial (In)Justice In Liberal Democratic Empire
Contemporary Voting Rights Controversies Through the Lens of Disability (reprint)
How Investors Can (and Can't) Create Social Value
Restatement (Third) of Torts: Miscellaneous Provisions: Preliminary Draft No. 5
Restatement (Third) of Torts: Miscellaneous Provisions: Tentative Draft No. 3
Restatement (Third) of Torts: Medical Malpractice: Tentative Draft No. 2
Rios Torrentosos:  La Identidad Personal en Tiempos Modernos [Torrential Rivers: Personal Identity in Modern Times]
How Investors Can (and Can't) Create Social Value
CRISPR People: The Science and Ethics of Editing Humans (Ita

In [ ]:
### Titles with multiple spaces
multi_space = titles[titles.str.contains(r'\s{2,}')].tolist()
print_list("Titles with MULTIPLE SPACES", multi_space)

### This code shows that some of titles have messy formatting.


===== Titles with MULTIPLE SPACES (count = 23) =====
Of One Mind and of One Government: The Rise and Fall of the Creek Nation  in the Early Republic
A Novel Model for Generating Creative,  Community-Responsive Interventions to Reduce Gender-Based Violence on College  Campuses
Rethinking the Lawyers' Monopoly:  Access to Justice and the Future of Legal Services
Legal Tech and the Litigation Playing  Field
Rethinking the Lawyers' Monopoly:  Access to Justice and the Future of Legal Services
That's the Way I Am: Lawrence M. Friedman in  Interview
Rios Torrentosos:  La Identidad Personal en Tiempos Modernos [Torrential Rivers: Personal Identity in Modern Times]
The Globalization of Mass Civil Litigation: Lessons from the Volkswagen  "Clean Diesel" Case
Evaluating Facial Recognition Technology: A Protocol for Performance  Assessment in New Domains
Enhancing Environmental Enforcement with Near Real-Time Monitoring:  Likelihood-Based Detection of Structural Expansion of Intensive Livestock F

In [ ]:
### CAPS titles
all_caps = titles[titles.str.fullmatch(r'[A-Z0-9\s\-\.,:;!?]+')].tolist()
print_list("Titles in ALL CAPS", all_caps)

#### This shows that some titles have CAPs. They need to be reviewed.


===== Titles in ALL CAPS (count = 5) =====
PROCESSES OF CONSTITUTIONAL DECISIONMAKING: CASES AND MATERIALS
WHITE-COLLAR CRIME
THE LAW AND ECONOMICS OF INTERNATIONAL TRADE AGREEMENTS
LIQUID ASSET: HOW BUSINESS AND GOVERNMENT CAN PARTNER TO SOLVE THE FRESHWATER CRISIS
EVALUATING THE USE OF DATA PLATFORMS FOR WATER MANAGEMENT DECISIONS


In [ ]:
### Titles with Unicode quotes
unicode_quotes = titles[titles.str.contains(r'[“”‘’]')].tolist()
print_list("Titles with UNICODE QUOTES", unicode_quotes)


===== Titles with UNICODE QUOTES (count = 3) =====
Donating Money To Help Your Child Get Into College Isn’t Wrong
Shining a Light on “Opaque Capital”
Unraveling the Truth of My Husband’s Stroke After His COVID Vaccine


In [ ]:
### very short titles (< 15 chars)
short_titles = titles[titles.str.len() < 15].tolist()
print_list("VERY SHORT Titles (<15 characters)", short_titles)


===== VERY SHORT Titles (<15 characters) (count = 37) =====
Book Review
Indigenous Law
Racism
Racial Borders
Book Review
Book Review
Border Wounds
Flores v. Reno
Moving Forward
Editor's Note
Editors' Note
Editors' Note
Lochner.com?
Oboe Judging
Bounty Regimes
The Hill
Evidence
On the Merits
Book Review
No Name
The Red Kimono
Book Review
Contracts 2023
Nature
Nature
Untitled Essay
Injuries
Injuries
Health Affairs
Video Game Law
Video Game Law
Exit Strategy
Fair Learning
Foreword
First Things
Actings
Introduction


In [ ]:
titles = df["Title"].astype(str)

### Find broken unicode artifacts
broken_unicode = titles[titles.str.contains(r"‚Ä|√|Ã", regex=True)]
print("Broken Unicode Count:", len(broken_unicode))
display(broken_unicode)


Broken Unicode Count: 0


,Title


In [ ]:
html_tags = titles[titles.str.contains(r"<[^>]+>", regex=True)]
print("Titles with HTML tags:", len(html_tags))

Titles with HTML tags: 58


In [ ]:
all_caps = titles[titles.str.fullmatch(r"[A-Z0-9\s\-\.,:;!?]+")]
print("ALL CAPS titles:", len(all_caps))
display(all_caps)


ALL CAPS titles: 5


,Title
132,PROCESSES OF CONSTITUTIONAL DECISIONMAKING: CASES AND MATERIALS
1318,WHITE-COLLAR CRIME
1778,THE LAW AND ECONOMICS OF INTERNATIONAL TRADE AGREEMENTS
1798,LIQUID ASSET: HOW BUSINESS AND GOVERNMENT CAN PARTNER TO SOLVE THE FRESHWATER CRISIS
1804,EVALUATING THE USE OF DATA PLATFORMS FOR WATER MANAGEMENT DECISIONS


In [ ]:
### long titles
long_titles = titles[titles.str.len() > 200]
print("Long titles (>200 chars):", len(long_titles))
display(long_titles)


Long titles (>200 chars): 14


,Title
212,"Brief for Amici Curiae Public Health Researchers and Social Scientists in Support Of Respondents, New York State Rifle & Pistol Association, et al., Petitioners, v. City of New York, et al., Respondents"
608,Points to Consider When Assessing Relationships (or Suspecting Misattributed Relationships) During Family-Based Clinical Genomic Testing: A Statement of the American College of Medical Genetics and Genomics (ACMG)
1020,"Brief Amici Curiae of 26 Intellectual Property Professors in Support of Granting the Petition for En Banc Review: Hologic, Inc., CYTYC Surgical Products, Plaintiffs-Appellants, v. Minerva Surgical, Inc., Defendant-Cross-Appellant"
1031,"Brief Amici Curiae of 23 Copyright Law Professors in Support of Defendant's Objection to Magistrate's Recommendation in Warner Records Inc., et al, Plaintiffs, v. Charter Communications Inc., Defendant"
1040,"Brief Amici Curiae of Intellectual Property Law Professors in Support of Appellees: Regents of the University of Minnesota v. LSI Corporation, Avago Technologies U.S. Inc., Gilead Sciences, Inc., Appellees"
1047,"Brief of Amici Curiae Law Professors and Public Knowledge in Support of Petitioners: EVE-USA, Inc., Synopsis Emulation and Verification, S.A.S., Synopsys, Inc., Petitioners v. Mentor Graphics Corporation"
1048,"Brief Amici Curiae of 37 Intellectual Property Law Professors in Support of Appellees' Petition for Panel Rehearing and Rehearing en Banc: Gordon v. Drape Creative,Inc. and Papyrus-Recycled Greetings, Inc., Appellees"
1075,"Brief <i>Amici Curiae</i> of 23 Copyright Law Professors in Support of Defendant's Objection to Magistrate's Recommendation in <i>Warner Records Inc., et al, Plaintiffs, v. Charter Communications Inc., Defendant</i>"
1078,"Brief <i>Amici Curiae</i> of 37 Intellectual Property Law Professors in Support of Appellees' Petition for Panel Rehearing and Rehearing en Banc: <i>Gordon v. Drape Creative,Inc. and Papyrus-Recycled Greetings, Inc., Appellees</i>"
1079,"Brief of <i>Amici Curiae</i> Law Professors and Public Knowledge in Support of Petitioners: <i>EVE-USA, Inc., Synopsis Emulation and Verification, S.A.S., Synopsys, Inc., Petitioners v. Mentor Graphics Corporation</i>"


In [ ]:
### legal case names
case_titles = titles[titles.str.contains(r"\bv\.\b", regex=True)]
print("Case-name titles:", len(case_titles))
display(case_titles)

Case-name titles: 0


,Title


In [ ]:
### titles with multiple subtitles: colon detection
multi_colon = titles[titles.str.count(":") >= 2]
print("Multiple-colon titles:", len(multi_colon))
display(multi_colon)

Multiple-colon titles: 22


,Title
48,"2021 Harrell-Bond Lecture, Refugee Studies Centre, Oxford: Empire's Refugee Studies Centre, Oxford: Empire's Refugees"
59,In Memoriam: Professor Gerald E. Frug The Messy Art of Public Freedom: A Tribute to Jerry Frug
151,"United States: The Amorphous Border: Deputization, Privatization, and Racialization"
181,Editors' Note: From the Editors: Left Elsewhere
182,Editors' Note: From the Editors: Racist Logic
331,Restatement (Third) of Torts: Miscellaneous Provisions: Preliminary Draft No. 5
335,Restatement (Third) of Torts: Miscellaneous Provisions: Tentative Draft No. 3
336,Restatement (Third) of Torts: Medical Malpractice: Tentative Draft No. 2
367,"Project Spotlight: Restatement of the Law Third, Torts: Concluding Provisions"
387,The New Wigmore: a Treatise on Evidence: the Right to Confrontation


In [ ]:
### checking duplicates
### not all the same titles are the same publications.
exact_dups = titles[titles.duplicated(keep=False)]
print("Exact duplicate titles:", len(exact_dups))
display(exact_dups)

Exact duplicate titles: 342


,Title
10,The Supreme Court Strikes Again — This Time at Tribal Sovereignty
13,Book Review
35,Selling Patents to Indian Tribes to Delay the Market Entry of Generic Drugs
64,Book Review
65,Local Government Law: Cases and Materials
78,Collecting the Rent: The Global Battle to Capture MNE Profits
81,Anxiety Psychoeducation for Law Students: A Pilot Program
86,Introduction to Special Issue
87,Stanford Law Faculty on the Historic Confirmation of Judge Ketanji Brown Jackson to the Supreme Court
90,When Silence Isn't Golden


In [ ]:
### Exact duplicates (case-sensitive)
### Now checking duplicates by author names and Publication type
dup_mask = df["Title"].duplicated(keep=False)

duplicate_titles_df = (
    df.loc[dup_mask, ["Title", "Faculty Contributor", "Year", "Publication Type"]]
    .sort_values("Title")
)

display(duplicate_titles_df)

,Title,Faculty Contributor,Year,Publication Type
1491,"2024 Supplement, The Law of Democracy, Legal Structure of the the Political Process 6th Edition","Persily, Nathaniel",2024.0,Textbook/Casebook
818,"2024 Supplement, The Law of Democracy, Legal Structure of the the Political Process 6th Edition","Karlan, Pamela S.",2024.0,Textbook/Casebook
1131,"<em>Small v. Memphis Light, Gas, & Water</em>: Petition for Writ of Certiorari (U.S. Supreme Court)","McConnell, Michael W.",2020.0,Brief
1674,"<em>Small v. Memphis Light, Gas, & Water</em>: Petition for Writ of Certiorari (U.S. Supreme Court)","Sonne, James A.",2020.0,Brief
822,A Pro-Feminist Life: Sherry Colb and Abortion Rights,"Karlan, Pamela S.",2024.0,Journal Article
811,A Pro-Feminist Life: Sherry Colb and Abortion Rights,"Karlan, Pamela S.",2024.0,Journal Article
824,A Pro-Feminist Life: Sherry Colb and Abortion Rights,"Karlan, Pamela S.",2023.0,Journal Article
932,A Sober Look at SPACs,"Klausner, Michael",2020.0,Blog Postings
922,A Sober Look at SPACs,"Klausner, Michael",2022.0,Journal Article
929,A Sober Look at SPACs,"Klausner, Michael",2021.0,Journal Article


In [ ]:
### HTML pollution problem

## Check Publication Venue Places

## Language Detection

In [ ]:
### non-English detection
non_ascii = titles[titles.apply(lambda x: any(ord(c)>127 for c in x))]
print("Non-ASCII titles:", len(non_ascii))
display(non_ascii)

### This doesn't work. LLMs could help.

Non-ASCII titles: 82


,Title
10,The Supreme Court Strikes Again — This Time at Tribal Sovereignty
12,Joe Manchin is wrong — D.C. statehood is constitutional
27,"Species of Sovereignty: Native Nationhood, the United States, and International Law, 1783–1795"
91,Donating Money To Help Your Child Get Into College Isn’t Wrong
130,How to Measure—And Manage—Performance
290,We Should Focus on — and Invest in — AI That Serves People Without Lawyers
332,We Should Focus on — and Invest in — AI That Serves People Without Lawyers
337,Shining a Light on “Opaque Capital”
481,"Existe Uma Cultura Jurídica Moderna?,"
503,Board 3.0 – An Introduction


In [ ]:
### Below, I get a language detection LLM from HuggingFace

model_name = "papluca/xlm-roberta-base-language-detection"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

lang_pipe = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    truncation=True,
    max_length=128,
    top_k=None
)


def detect_language(title: str):
    """
    Robust Hugging Face language detection:
    Returns 'English', 'Non-English', or 'Unknown'
    """
    if pd.isna(title) or str(title).strip() == "":
        return "Unknown"

    # run model
    result = lang_pipe(title)

    if isinstance(result, list) and len(result) == 1 and isinstance(result[0], list):
        result = result[0]
    if not isinstance(result, list) or not isinstance(result[0], dict):
        return "Unknown"
    best = max(result, key=lambda x: x.get("score", 0))

    lang = best.get("label", "").lower()

    if lang == "en":
        return "English"
    elif lang == "":
        return "Unknown"
    else:
        return "Non-English"

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/502 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Device set to use cpu


In [ ]:
tqdm.pandas()
df["Language_HF"] = df["Title"].progress_apply(detect_language)


100%|██████████| 1927/1927 [04:17<00:00,  7.48it/s]


In [ ]:
df["Language_HF"].value_counts()

,count
Language_HF,
English,1835
Non-English,92


In [ ]:
df_non_english = df[df["Language_HF"] == "Non-English"]
df_non_english


### titles needs to be cleaned before language detection

,ID,Faculty Contributor,Faculty Co-Authors/Editors,Co-Authors,Title,Publication Type,Contribution,Date,Year,Status,...,Source Link(s),Stanford Link,DOI,PURL,SSRN,In ORCID,Corrected,Designation,Language_HF,LanguageLLM
4,407547.0,"Ablavsky, Gregory",NaN,NaN,Akhil Amar's Unusable Past,Book Review,Writer,2023,2023.0,Published,...,NaN,https://law.stanford.edu/publications/akhil-amars-unusable-past/,NaN,NaN,NaN,NaN,checked,Faculty,Non-English,Uncertain
9,407549.0,"Ablavsky, Gregory",NaN,NaN,Oklahoma's Bizarro Nineteenth Century in Castro-Huerta,Blog Postings,Writer,2022-04-26,2022.0,Published,...,https://law.stanford.edu/2022/04/26/oklahomas-bizarro-nineteenth-century-in-castro-huerta/,https://law.stanford.edu/publications/oklahomas-bizarro-nineteenth-century-in-castro-huerta/,NaN,NaN,NaN,NaN,checked,Faculty,Non-English,Uncertain
30,308303.0,"Ablavsky, Gregory",NaN,Sarah Deer; Justin Richland,Indigenous Law,"Book, Section",Writer,2019-12-01,2019.0,Published,...,https://global.oup.com/academic/product/the-oxford-handbook-of-law-and-humanities-9780190695620?cc=us&lang=en&#,https://law.stanford.edu/publications/indigenous-law/,NaN,NaN,NaN,NaN,checked,Faculty,Non-English,Uncertain
38,496612.0,"Achiume, E. Tendayi",NaN,NaN,Racism,"Book, Section",Writer,2025,2025.0,Forthcoming,...,NaN,https://law.stanford.edu/publications/racism-in-elgar-concise-encyclopedia-of-migration-law/,NaN,NaN,NaN,NaN,checked,Faculty,Non-English,Uncertain
52,498688.0,"Achiume, E. Tendayi",NaN,NaN,Transnational Racial (In)Justice In Liberal Democratic Empire,Journal Article,Writer,2021,2021.0,Published,...,NaN,https://law.stanford.edu/publications/transnational-racial-injustice-in-liberal-democratic-empire/,NaN,NaN,NaN,NaN,checked,Faculty,Non-English,Uncertain
55,499691.0,"Achiume, E. Tendayi",NaN,NaN,Governing Xenophobia,Journal Article,Writer,2021,2021.0,Published,...,NaN,https://law.stanford.edu/publications/governing-xenophobia/,NaN,NaN,NaN,NaN,checked,Faculty,Non-English,Uncertain
139,448263.0,"Brodie, Juliet M.",NaN,NaN,Lawyers Aren't Rent,Journal Article,Writer,2023,2023.0,Published,...,NaN,https://law.stanford.edu/publications/lawyers-arent-rent/,NaN,NaN,NaN,NaN,NaN,Clinical Faculty,Non-English,Uncertain
147,499867.0,"Campbell, Jud",NaN,NaN,Constitutional Rights,Textbook/Casebook,NaN,2024-07,2024.0,Published,...,NaN,https://law.stanford.edu/publications/constitutional-rights-3/,NaN,NaN,NaN,NaN,checked,Faculty,Non-English,Uncertain
161,448269.0,"Chacón, Jennifer M.",NaN,NaN,Border Wounds,Blog Postings,Writer,2022-09,2022.0,Published,...,NaN,https://law.stanford.edu/publications/border-wounds/,NaN,NaN,NaN,NaN,checked,Faculty,Non-English,Uncertain
164,403984.0,"Chacón, Jennifer M.",NaN,NaN,Flores v. Reno,"Book, Section",Writer,2022-05,2022.0,Published,...,https://www.cambridge.org/core/books/critical-race-judgments/50FD6F8BD13E9D09B57D9FEE3040CA24,https://law.stanford.edu/publications/flores-v-reno/,NaN,NaN,NaN,NaN,checked,Faculty,Non-English,Uncertain


In [ ]:
### LLMs are good at detecting language as they have seen all languages available online.
### I didn't connect API because of data privacy issues.
### Also, I think it is possible to get a local based LLM to do this to 1) avoid cost, 2) have full control
### Still would be good to compare models
openai.api_key = "API_Key"

def classify_title_language(title: str) -> str:
    """
    Uses an LLM to classify whether a title is English or non-English.
    Returns: "English", "Non-English", or "Uncertain".
    """
    prompt = f"""
    Determine whether the following academic publication title is written in English or in a non-English language.

    Title: {title}

    Respond with ONLY one word:
    - English
    - Non-English
    - Uncertain (only if genuinely ambiguous)
    """

    try:
        response = openai.ChatCompletion.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )

        result = response["choices"][0]["message"]["content"].strip()

        if "english" in result.lower():
            return "English"
        elif "non" in result.lower():
            return "Non-English"
        else:
            return "Uncertain"

    except Exception as e:
        print("Error:", e)
        return "Uncertain"

tqdm.pandas()
df["LanguageLLM"] = df["Title"].progress_apply(classify_title_language)

### Display sample results
display(df[["Title", "LanguageLLM"]].head(20))

### Save
df.to_csv("titles_with_language_llm.csv", index=False)